## 📊 Part 3 — PCA for High-Dimensional Business Data: Retail Transactions

Real customer analytics often involves dozens of features: purchase frequency across categories,
recency scores, channel preferences, seasonal indices. PCA compresses these into a handful
of interpretable dimensions.

In [ ]:
# Synthetic retail customer feature matrix
# 500 customers × 20 features: category spend, frequency, recency, channel mix
np.random.seed(7)
n_customers = 500

# Underlying latent dimensions
price_sensitivity  = np.random.normal(0, 1, n_customers)  # PC1
engagement         = np.random.normal(0, 1, n_customers)  # PC2
channel_pref       = np.random.normal(0, 1, n_customers)  # PC3

features = {
    'grocery_spend':     2*engagement + price_sensitivity*0.5 + np.random.normal(0,.5,n_customers),
    'electronics_spend': engagement - price_sensitivity + np.random.normal(0,.6,n_customers),
    'clothing_spend':    engagement*0.8 + channel_pref*0.4 + np.random.normal(0,.5,n_customers),
    'home_spend':        engagement*0.6 + np.random.normal(0,.7,n_customers),
    'beauty_spend':      engagement*0.7 - channel_pref*0.3 + np.random.normal(0,.4,n_customers),
    'frequency':         engagement*1.2 + np.random.normal(0,.4,n_customers),
    'recency_days':      -engagement*0.8 + np.random.normal(0,.5,n_customers),
    'avg_basket':        engagement*0.5 - price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'returns_rate':      price_sensitivity*0.3 + np.random.normal(0,.4,n_customers),
    'promo_sensitivity': price_sensitivity*1.5 + np.random.normal(0,.3,n_customers),
    'mobile_pct':        channel_pref*1.2 + np.random.normal(0,.4,n_customers),
    'email_open_rate':   engagement*0.4 + channel_pref*0.5 + np.random.normal(0,.5,n_customers),
    'loyalty_points':    engagement*1.0 + np.random.normal(0,.4,n_customers),
    'weekend_pct':       channel_pref*0.3 + np.random.normal(0,.6,n_customers),
    'private_label_pct': price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'cross_category':    engagement*0.9 + np.random.normal(0,.4,n_customers),
    'seasonal_index':    np.random.normal(0,1,n_customers),
    'store_vs_online':   channel_pref*1.0 + np.random.normal(0,.4,n_customers),
    'referral_flag':     engagement*0.3 + np.random.normal(0,.8,n_customers),
    'complaint_count':  -engagement*0.4 + price_sensitivity*0.2 + np.random.normal(0,.5,n_customers),
}
X_retail = pd.DataFrame(features)
print(f"Retail customer feature matrix: {X_retail.shape}")
print(f"\n20 features per customer — hard to visualize or model directly")
print(f"PCA will find the key dimensions driving customer variation")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize (features are on different scales — spend vs rates vs counts)
scaler_r = StandardScaler()
X_scaled_r = scaler_r.fit_transform(X_retail)

# Fit PCA
pca_retail = PCA()
X_pca_r = pca_retail.fit_transform(X_scaled_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
cumvar = np.cumsum(pca_retail.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1

axes[0].bar(range(1,21), pca_retail.explained_variance_ratio_*100,
            color='#1e5fa8', edgecolor='white', alpha=0.8, label='Individual')
axes[0].plot(range(1,21), cumvar*100, 'o-', color='#e85d20', lw=2,
             markersize=5, label='Cumulative')
axes[0].axhline(90, color='#888', lw=1, ls='--', alpha=0.6, label='90% threshold')
axes[0].axvline(n_90, color='#1a7a45', lw=1.5, ls='--',
                label=f'{n_90} PCs explain 90%')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('How Many Dimensions Capture Retail Customer Variation?')
axes[0].legend(fontsize=8)

# PC1 vs PC2 scatter — colored by engagement
axes[1].scatter(X_pca_r[:,0], X_pca_r[:,1],
                c=features['frequency'], cmap='RdYlGn', s=20, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_retail.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca_retail.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('Customers in 2D PCA Space\n(color = purchase frequency)')
sm = plt.cm.ScalarMappable(cmap='RdYlGn')
plt.colorbar(sm, ax=axes[1], label='Purchase Frequency', shrink=0.8)

plt.tight_layout(); plt.show()
print(f"\n\u2714 {n_90} components capture 90% of customer variation (vs 20 original features)")
print(f"   Compression ratio: {20/n_90:.1f}x — huge simplification for downstream modeling")

In [ ]:
# What does each PC represent? — Loadings analysis
loadings = pd.DataFrame(pca_retail.components_[:3].T,
                        index=X_retail.columns,
                        columns=['PC1','PC2','PC3'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pc in zip(axes, ['PC1','PC2','PC3']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#e85d20' if v > 0 else '#1e5fa8' for v in sorted_load]
    ax.barh(sorted_load.index, sorted_load.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{pc} Loadings\n({pca_retail.explained_variance_ratio_[int(pc[-1])-1]*100:.1f}% variance)')
    ax.set_xlabel('Loading')

plt.suptitle('PCA Loadings: What Does Each Component Represent?\n'
             'Orange = positive contribution, Blue = negative',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()
print("\n\u2714 PC1: high loadings on frequency, loyalty, cross-category = 'Engagement'")
print("   PC2: high loadings on promo sensitivity, private label = 'Price Sensitivity'")
print("   PC3: high loadings on mobile, store_vs_online = 'Channel Preference'")
print("   These match the latent dimensions we built into the data!")

In [ ]:
answers = {
    "q1": "",  # What does the first principal component maximize?
    "q2": "",  # What do loadings tell you about a principal component?
    "q3": "",  # Why must features be standardized before PCA?
    "q4": "",  # You have 100 features. The first 5 PCs explain 85% of variance. What would you do next?
    "q5": "",  # What is one limitation of PCA for classification tasks?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
# ── Submit your results to the instructor ────────────────────────────────────
# Fill in your GitHub username (do this once, it stays saved in the notebook)
GITHUB_USERNAME = ""   # ← put your GitHub username here e.g. "jsmith42"

# ─────────────────────────────────────────────────────────────────────────────
# DO NOT EDIT BELOW THIS LINE
import json, urllib.parse

def build_submission_url(username, notebook_name, score, total, answers_dict,
                          form_base_url=None, entries=None):
    """Build a pre-filled Google Form URL for quiz submission."""
    if not form_base_url:
        print("⚠  No submission URL configured yet.")
        print("   Your instructor will share the Form URL — paste it into form_base_url below.")
        print("   Your answers are saved in this notebook regardless.")
        return None

    params = {
        entries.get('user',     'entry.000000001'): username,
        entries.get('notebook', 'entry.000000002'): notebook_name,
        entries.get('score',    'entry.000000003'): str(score),
        entries.get('total',    'entry.000000004'): str(total),
    }
    url = form_base_url.replace('/viewform','') + '/viewform?' + urllib.parse.urlencode(params)
    return url

# ── Submission config (your instructor fills these in) ───────────────────────
FORM_BASE_URL = None   # e.g. "https://docs.google.com/forms/d/e/XXXX/viewform"
FORM_ENTRIES  = {      # entry IDs from the Google Form
    'user':     'entry.000000001',
    'notebook': 'entry.000000002',
    'score':    'entry.000000003',
    'total':    'entry.000000004',
}
# ─────────────────────────────────────────────────────────────────────────────

# Calculate score from answers dict
score_val = sum(1 for v in answers.values() if str(v).strip())  # counts answered Qs
# For auto-graded notebooks the score is passed directly
# Here we just verify completion
n_answered = sum(1 for v in answers.values() if str(v).strip())
n_total    = len(answers)

print(f"\n{'='*50}")
print(f"Quiz completion: {n_answered}/{n_total} questions answered")
if n_answered < n_total:
    print(f"⚠  Please answer all {n_total} questions before submitting.")
else:
    print("✅ All questions answered!")
    if GITHUB_USERNAME:
        import os
        nb_name = os.path.basename(globals().get('__vsc_ipynb_file__',
                  globals().get('__file__', 'unknown-notebook'))).replace('.ipynb','')
        url = build_submission_url(GITHUB_USERNAME, nb_name,
                                   n_answered, n_total, answers,
                                   FORM_BASE_URL, FORM_ENTRIES)
        if url:
            print(f"\n🔗 Submit to instructor:")
            print(f"   {url}")
            print("\n   Copy the URL above, open it in a new tab, and click Submit.")
        else:
            print("\n   Save your notebook to GitHub to record your progress:")
            print("   File → Save a copy in GitHub → choose your fork")
    else:
        print("\n⚠  Add your GitHub username to GITHUB_USERNAME above, then re-run.")
    print(f"{'='*50}")